In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
RAW_DIR   = os.path.join(DATA_ROOT, 'raw')
TEST_DIR  = os.path.join(DATA_ROOT, 'test_in')
OUT_PATH  = '/kaggle/working/preds.npy'

LOOKBACK   = 10
HORIZON    = 16
H, W       = 140, 124
BATCH_SIZE = 4
EPOCHS     = 80
LR         = 2e-4
STRIDE     = 2
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MET_FEATS  = ['q2','t2','u10','v10','swdown','pblh','psfc','rain']
EMIT_FEATS = ['PM25','NH3','SO2','NOx','NMVOC_e','NMVOC_finn','bio']
MONTHS     = ['APRIL_16','JULY_16','OCT_16','DEC_16']

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device : cuda
PyTorch: 2.10.0+cu128


In [3]:
def load_month(month):
    path = os.path.join(RAW_DIR, month)
    data = {}
    data['cpm25'] = np.load(os.path.join(path,'cpm25.npy')).astype(np.float32)
    for f in MET_FEATS + EMIT_FEATS:
        fp = os.path.join(path, f'{f}.npy')
        if os.path.exists(fp):
            data[f] = np.load(fp).astype(np.float32)
    return data

print("Loading training months...")
month_data = {}
for m in MONTHS:
    month_data[m] = load_month(m)
    print(f"  {m}: T={month_data[m]['cpm25'].shape[0]}, "
          f"features={list(month_data[m].keys())}")

Loading training months...
  APRIL_16: T=715, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
  JULY_16: T=739, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
  OCT_16: T=739, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
  DEC_16: T=739, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']


In [4]:
def engineer_features_train(data_dict):
    """
    For training monthly arrays shape (T, H, W).
    Only features computable within a single 10h window — safe for test.
    """
    d = dict(data_dict)

    # Wind — safe, computed per timestep
    if 'u10' in d and 'v10' in d:
        d['wind_speed'] = np.sqrt(d['u10']**2 + d['v10']**2)
        d['wind_dir']   = np.arctan2(d['v10'], d['u10'])

    # Boundary layer inverse — safe, per timestep
    if 'pblh' in d:
        pblh          = np.where(d['pblh'] < 1.0, 1.0, d['pblh'])
        d['inv_pblh'] = 1.0 / pblh

    # Rain log — safe, per timestep
    if 'rain' in d:
        d['log_rain'] = np.log1p(d['rain'])

    # Ventilation coefficient — safe, per timestep
    if 'wind_speed' in d and 'pblh' in d:
        d['vent_coeff'] = d['wind_speed'] * d['pblh'] / 10000.0

    return d


def engineer_features_test(data_dict):
    """
    For test arrays shape (N, 10, H, W).
    Exact same operations — works because all are per-timestep.
    """
    d = dict(data_dict)

    if 'u10' in d and 'v10' in d:
        d['wind_speed'] = np.sqrt(d['u10']**2 + d['v10']**2)
        d['wind_dir']   = np.arctan2(d['v10'], d['u10'])

    if 'pblh' in d:
        pblh          = np.where(d['pblh'] < 1.0, 1.0, d['pblh'])
        d['inv_pblh'] = 1.0 / pblh

    if 'rain' in d:
        d['log_rain'] = np.log1p(d['rain'])

    if 'wind_speed' in d and 'pblh' in d:
        d['vent_coeff'] = d['wind_speed'] * d['pblh'] / 10000.0

    return d


for m in MONTHS:
    month_data[m] = engineer_features_train(month_data[m])
    print(f"  {m}: {len(month_data[m])} features: {list(month_data[m].keys())}")

  APRIL_16: 21 features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir', 'inv_pblh', 'log_rain', 'vent_coeff']
  JULY_16: 21 features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir', 'inv_pblh', 'log_rain', 'vent_coeff']
  OCT_16: 21 features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir', 'inv_pblh', 'log_rain', 'vent_coeff']
  DEC_16: 21 features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir', 'inv_pblh', 'log_rain', 'vent_coeff']


In [5]:
BASE_FEATS  = ['cpm25'] + MET_FEATS + EMIT_FEATS
# Only sample-safe engineered features
EXTRA_FEATS = ['wind_speed','wind_dir','inv_pblh','log_rain','vent_coeff']

FEAT_KEYS = []
for f in BASE_FEATS + EXTRA_FEATS:
    if all(f in month_data[m] for m in MONTHS):
        if f not in FEAT_KEYS:
            FEAT_KEYS.append(f)

# LOCK
N_FEATS = len(FEAT_KEYS)
print(f"FEAT_KEYS locked: {N_FEATS} features")
print(FEAT_KEYS)

def compute_norm_stats(month_data, feat_keys):
    stats = {}
    for feat in feat_keys:
        combined = np.concatenate(
            [month_data[m][feat] for m in MONTHS], axis=0)
        med = np.median(combined, axis=0)
        iqr = (np.percentile(combined, 75, axis=0)
             - np.percentile(combined, 25, axis=0))
        iqr = np.where(iqr < 1e-6, 1.0, iqr)
        stats[feat] = (med.astype(np.float32), iqr.astype(np.float32))
        del combined
        gc.collect()
    return stats

print("Computing normalization stats...")
norm_stats           = compute_norm_stats(month_data, FEAT_KEYS)
cpm25_med, cpm25_iqr = norm_stats['cpm25']

def normalize_month(data_dict, stats, feat_keys):
    arrays = []
    for feat in feat_keys:
        med, iqr = stats[feat]
        arrays.append(((data_dict[feat] - med) / iqr).astype(np.float32))
    stacked = np.stack(arrays, axis=-1)
    del arrays
    return stacked

norm_data = {}
for m in MONTHS:
    norm_data[m] = normalize_month(month_data[m], norm_stats, FEAT_KEYS)
    print(f"  {m}: {norm_data[m].shape} "
          f"({norm_data[m].nbytes/1024**2:.0f} MB)")

del month_data
gc.collect()
total_mb = sum(v.nbytes for v in norm_data.values()) / 1024**2
print(f"\nTotal norm_data RAM: {total_mb:.0f} MB | N_FEATS={N_FEATS}")

FEAT_KEYS locked: 21 features
['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir', 'inv_pblh', 'log_rain', 'vent_coeff']
Computing normalization stats...
  APRIL_16: (715, 140, 124, 21) (994 MB)
  JULY_16: (739, 140, 124, 21) (1028 MB)
  OCT_16: (739, 140, 124, 21) (1028 MB)
  DEC_16: (739, 140, 124, 21) (1028 MB)

Total norm_data RAM: 4077 MB | N_FEATS=21


In [6]:
class PM25Dataset(Dataset):
    def __init__(self, norm_arrays, lookback=LOOKBACK,
                 horizon=HORIZON, stride=STRIDE):
        self.lookback = lookback
        self.horizon  = horizon
        self.arrays   = norm_arrays
        self.cpm_idx  = FEAT_KEYS.index('cpm25')
        self.indices  = []
        self.weights  = []

        for arr_idx, arr in enumerate(norm_arrays):
            T      = arr.shape[0]
            window = lookback + horizon
            for i in range(0, T - window + 1, stride):
                self.indices.append((arr_idx, i))
                target_win = arr[i+lookback:i+window,
                                 :, :, self.cpm_idx]
                w = float(np.percentile(target_win, 90))
                self.weights.append(max(w, 0.01))

        self.weights = np.array(self.weights, dtype=np.float32)
        self.weights = self.weights / self.weights.sum()
        print(f"  Samples: {len(self.indices)}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        arr_idx, t = self.indices[idx]
        arr = self.arrays[arr_idx]
        x   = arr[t : t + self.lookback].copy()
        y   = arr[t + self.lookback : t + self.lookback + self.horizon,
                  :, :, self.cpm_idx].copy()
        return torch.from_numpy(x).float(), torch.from_numpy(y).float()


dec_arr   = norm_data['DEC_16']
dec_train = dec_arr[:-300]
dec_val   = dec_arr[-300:]

print("Train:")
train_ds = PM25Dataset(
    [norm_data['APRIL_16'], norm_data['JULY_16'],
     norm_data['OCT_16'],   dec_train],
    stride=STRIDE)

print("Val:")
val_ds = PM25Dataset([dec_val], stride=2)

sampler = WeightedRandomSampler(
    weights     = torch.from_numpy(train_ds.weights),
    num_samples = len(train_ds),
    replacement = True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train:
  Samples: 1266
Val:
  Samples: 138
Train batches: 317 | Val batches: 35


In [7]:
class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, hidden), nn.ReLU(),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )
        self.spatial_att = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3), nn.Sigmoid()
        )
    def forward(self, x):
        ca  = self.channel_att(x).view(x.shape[0], x.shape[1], 1, 1)
        x   = x * ca
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1,  keepdim=True)[0]
        return x * self.spatial_att(torch.cat([avg, mx], dim=1))


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel=3):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.gates     = nn.Conv2d(
            in_ch + hidden_ch, 4*hidden_ch, kernel, padding=kernel//2)
    def forward(self, x, h, c):
        gates    = self.gates(torch.cat([x, h], dim=1))
        i, f, g, o = gates.chunk(4, dim=1)
        c_new    = torch.sigmoid(f)*c + torch.sigmoid(i)*torch.tanh(g)
        h_new    = torch.sigmoid(o)*torch.tanh(c_new)
        return h_new, c_new
    def init_hidden(self, B, H, W, device):
        return (torch.zeros(B, self.hidden_ch, H, W, device=device),
                torch.zeros(B, self.hidden_ch, H, W, device=device))


class DualPathEncoder(nn.Module):
    def __init__(self, n_feats, hidden):
        super().__init__()
        self.proj_a = nn.Conv2d(n_feats, hidden, 1)
        self.lstm   = ConvLSTMCell(hidden, hidden)
        self.proj_b = nn.Conv2d(LOOKBACK * n_feats, hidden, 1)
        self.fuse   = nn.Sequential(
            nn.Conv2d(hidden*2, hidden, 1),
            nn.BatchNorm2d(hidden), nn.GELU()
        )
    def forward(self, x):
        B, T, H, W, C = x.shape
        h, c = self.lstm.init_hidden(B, H, W, x.device)
        for t in range(T):
            xt   = x[:, t].permute(0,3,1,2)
            h, c = self.lstm(self.proj_a(xt), h, c)
        out_a = h
        flat  = x.permute(0,1,4,2,3).reshape(B, T*C, H, W)
        out_b = self.proj_b(flat)
        return self.fuse(torch.cat([out_a, out_b], dim=1))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_attn=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )
        self.attn = CBAM(out_ch) if use_attn else nn.Identity()
        self.res  = (nn.Conv2d(in_ch, out_ch, 1)
                     if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        return self.attn(self.conv(x)) + self.res(x)


class PM25Net(nn.Module):
    def __init__(self, n_feats, horizon, base=48):
        super().__init__()
        self.horizon    = horizon
        self.enc_in     = DualPathEncoder(n_feats, base)
        self.enc1       = ConvBlock(base,    base*2, False)
        self.enc2       = ConvBlock(base*2,  base*4, False)
        self.enc3       = ConvBlock(base*4,  base*8, True)
        self.pool       = nn.MaxPool2d(2, ceil_mode=True)
        self.bottleneck = ConvBlock(base*8,  base*16, True)
        self.up3        = nn.ConvTranspose2d(base*16, base*8,  2, stride=2)
        self.dec3       = ConvBlock(base*16, base*8,  True)
        self.up2        = nn.ConvTranspose2d(base*8,  base*4,  2, stride=2)
        self.dec2       = ConvBlock(base*8,  base*4,  False)
        self.up1        = nn.ConvTranspose2d(base*4,  base*2,  2, stride=2)
        self.dec1       = ConvBlock(base*4,  base*2,  False)
        # Split horizon head
        half            = horizon // 2
        self.head_near  = nn.Sequential(
            nn.Conv2d(base*2, base, 3, padding=1), nn.GELU(),
            nn.Conv2d(base, half, 1)
        )
        self.head_far   = nn.Sequential(
            nn.Conv2d(base*2, base,  3, padding=1), nn.GELU(),
            nn.Conv2d(base,   base,  3, padding=1), nn.GELU(),
            nn.Conv2d(base,   half,  1)
        )

    def _match(self, x, ref):
        dh = ref.shape[2] - x.shape[2]
        dw = ref.shape[3] - x.shape[3]
        if dh > 0 or dw > 0:
            x = F.pad(x, [0, max(dw,0), 0, max(dh,0)])
        if x.shape[2] > ref.shape[2] or x.shape[3] > ref.shape[3]:
            x = x[:, :, :ref.shape[2], :ref.shape[3]]
        return x

    def forward(self, x):
        e0 = self.enc_in(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bn = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self._match(self.up3(bn), e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._match(self.up2(d3), e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._match(self.up1(d2), e1), e1], dim=1))
        near = self.head_near(d1)
        far  = self.head_far(d1)
        return torch.cat([near, far], dim=1)


print(f"Building model: N_FEATS={N_FEATS}, base=48")
model    = PM25Net(n_feats=N_FEATS, horizon=HORIZON, base=48).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    free_gb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) / 1024**3
    print(f"GPU free: {free_gb:.1f} GB")

with torch.no_grad():
    dummy = torch.zeros(2, LOOKBACK, H, W, N_FEATS).to(DEVICE)
    out   = model(dummy)
    assert out.shape == (2, HORIZON, H, W), f"Bad shape: {out.shape}"
    print(f"Shape check passed: {out.shape} ✓")
    del dummy
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

Building model: N_FEATS=21, base=48
Parameters: 18,653,161
GPU free: 14.5 GB
Shape check passed: torch.Size([2, 16, 140, 124]) ✓


In [8]:
def smape_loss(pred, target, eps=1.0):
    return torch.mean(
        torch.abs(pred - target) /
        (0.5*(torch.abs(target) + torch.abs(pred)) + eps)
    )

def pearson_corr_loss(pred, target):
    pf = pred.reshape(pred.shape[0], -1)
    tf = target.reshape(target.shape[0], -1)
    pm = pf - pf.mean(dim=1, keepdim=True)
    tm = tf - tf.mean(dim=1, keepdim=True)
    corr = (pm*tm).sum(dim=1) / (pm.norm(dim=1)*tm.norm(dim=1) + 1e-8)
    return 1.0 - corr.mean()

def episode_mask_batch(target, sigma=2.5):
    B, T, H, W = target.shape
    flat = target.view(B, T, -1)
    mean = flat.mean(dim=1, keepdim=True)
    std  = flat.std(dim=1,  keepdim=True)
    return ((flat - mean) > sigma * std).view(B, T, H, W)

def episode_aware_loss(pred, target,
                       w_global=0.25, w_ep_smape=0.40, w_ep_corr=0.35):
    loss_global = smape_loss(pred, target)
    mask        = episode_mask_batch(target)

    if mask.sum() > 50:
        loss_ep_smape = smape_loss(pred[mask], target[mask])
        loss_ep_corr  = pearson_corr_loss(
            pred*mask.float(), target*mask.float())
    else:
        loss_ep_smape = loss_global
        loss_ep_corr  = torch.tensor(0.0, device=pred.device)

    # Horizon weighting — penalise far steps more
    B, T2, Hx, Wx = pred.shape
    hw     = torch.linspace(0.8, 1.2, T2).to(pred.device).view(1,T2,1,1)
    loss_h = smape_loss(pred*hw, target*hw)

    total = (w_global   * loss_global
           + w_ep_smape * loss_ep_smape
           + w_ep_corr  * loss_ep_corr
           + 0.10       * loss_h)

    return total, loss_global.item(), loss_ep_smape.item(), loss_ep_corr.item()

print("Loss functions ready ✓")

Loss functions ready ✓


In [9]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=15, T_mult=2, eta_min=1e-6)

best_val_loss = float('inf')
best_ckpt     = '/kaggle/working/best_model.pt'
patience      = 20
no_improve    = 0

for epoch in range(1, EPOCHS+1):

    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        if pred.shape[-2:] != yb.shape[-2:]:
            pred = F.interpolate(pred, size=yb.shape[-2:],
                                 mode='bilinear', align_corners=False)
        loss, *_ = episode_aware_loss(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)
    scheduler.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred   = model(xb)
            if pred.shape[-2:] != yb.shape[-2:]:
                pred = F.interpolate(pred, size=yb.shape[-2:],
                                     mode='bilinear', align_corners=False)
            loss, *_ = episode_aware_loss(pred, yb)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt)
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 10 == 0 and DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}/{EPOCHS} | "
              f"Train: {tr_loss:.4f} | Val: {val_loss:.4f} | "
              f"Best: {best_val_loss:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e} | "
              f"Patience: {no_improve}/{patience}")

    if no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nDone. Best val loss: {best_val_loss:.4f}")

Epoch 001/80 | Train: 0.4218 | Val: 0.3281 | Best: 0.3281 | LR: 1.98e-04 | Patience: 0/20
Epoch 005/80 | Train: 0.1885 | Val: 0.2771 | Best: 0.2614 | LR: 1.50e-04 | Patience: 2/20
Epoch 010/80 | Train: 0.1427 | Val: 0.2071 | Best: 0.2071 | LR: 5.08e-05 | Patience: 0/20
Epoch 015/80 | Train: 0.1219 | Val: 0.2139 | Best: 0.1855 | LR: 2.00e-04 | Patience: 1/20
Epoch 020/80 | Train: 0.1313 | Val: 0.2682 | Best: 0.1855 | LR: 1.87e-04 | Patience: 6/20
Epoch 025/80 | Train: 0.1107 | Val: 0.7912 | Best: 0.1855 | LR: 1.50e-04 | Patience: 11/20
Epoch 030/80 | Train: 0.0921 | Val: 0.3475 | Best: 0.1855 | LR: 1.01e-04 | Patience: 16/20

Early stopping at epoch 34

Done. Best val loss: 0.1855


In [10]:
print("Loading test data...")
test_data = {}
test_data['cpm25'] = np.load(
    os.path.join(TEST_DIR, 'cpm25.npy')).astype(np.float32)
for f in MET_FEATS:
    fp = os.path.join(TEST_DIR, f'{f}.npy')
    if os.path.exists(fp):
        test_data[f] = np.load(fp).astype(np.float32)
for f in EMIT_FEATS:
    fp = os.path.join(TEST_DIR, f'{f}.npy')
    if os.path.exists(fp):
        test_data[f] = np.load(fp).astype(np.float32)

# Use test-safe engineer function
test_data = engineer_features_test(test_data)
N_TEST    = test_data['cpm25'].shape[0]
missing   = [f for f in FEAT_KEYS if f not in test_data]
print(f"Test samples : {N_TEST}")
print(f"Missing feats: {missing if missing else 'None ✓'}")

Loading test data...
Test samples : 218
Missing feats: None ✓


In [11]:
def build_test_tensor_single(sample_idx):
    arrays = []
    for feat in FEAT_KEYS:
        med, iqr = norm_stats[feat]
        if feat in test_data:
            arr    = test_data[feat][sample_idx]   # (10, H, W)
            normed = (arr - med[None]) / iqr[None]
        else:
            normed = np.zeros((LOOKBACK, H, W), dtype=np.float32)
        arrays.append(normed)
    return np.stack(arrays, axis=-1).astype(np.float32)

# Channel check
sample      = build_test_tensor_single(0)
expected_ch = model.enc_in.proj_a.weight.shape[1]
actual_ch   = sample.shape[-1]
assert actual_ch == expected_ch, \
    f"Channel mismatch! Model={expected_ch}, got={actual_ch}"
print(f"Channel check : {actual_ch} == {expected_ch} ✓")
print(f"Sample shape  : {sample.shape} ✓")
del sample

Channel check : 21 == 21 ✓
Sample shape  : (10, 140, 124, 21) ✓


In [12]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print("Best checkpoint loaded ✓")

preds_list = []
n_batches  = (N_TEST + BATCH_SIZE - 1) // BATCH_SIZE

with torch.no_grad():
    for i in range(0, N_TEST, BATCH_SIZE):
        batch_idx = range(i, min(i + BATCH_SIZE, N_TEST))
        batch_np  = np.stack(
            [build_test_tensor_single(j) for j in batch_idx], axis=0)
        xb = torch.from_numpy(batch_np).to(DEVICE)

        pred1 = model(xb)
        pred2 = torch.flip(model(torch.flip(xb, dims=[3])), dims=[3])
        pred3 = torch.flip(model(torch.flip(xb, dims=[2])), dims=[2])
        pred  = (pred1 + pred2 + pred3) / 3.0
        del pred1, pred2, pred3, xb

        if pred.shape[-2:] != (H, W):
            pred = F.interpolate(pred, size=(H, W),
                                 mode='bilinear', align_corners=False)

        pred_np = pred.cpu().numpy()
        del pred
        pred_np = pred_np * cpm25_iqr[None, None] + cpm25_med[None, None]
        pred_np = np.clip(pred_np, 0, None)
        preds_list.append(pred_np)
        print(f"  Batch {i//BATCH_SIZE+1}/{n_batches} done", end='\r')

preds = np.concatenate(preds_list, axis=0)
preds = preds.transpose(0, 2, 3, 1)
print(f"\nFinal shape: {preds.shape}")
assert preds.shape == (218, 140, 124, 16), f"Shape mismatch: {preds.shape}"
print("Shape check passed ✓")

Best checkpoint loaded ✓
  Batch 55/55 done
Final shape: (218, 140, 124, 16)
Shape check passed ✓


In [13]:
np.save(OUT_PATH, preds)
loaded = np.load(OUT_PATH)
print(f"Saved  → {OUT_PATH}")
print(f"Shape  : {loaded.shape}")
print(f"dtype  : {loaded.dtype}")
print(f"min/max: {loaded.min():.4f} / {loaded.max():.4f}")
print(f"NaNs   : {np.isnan(loaded).sum()}")
print(f"Negs   : {(loaded < 0).sum()}")
assert loaded.shape == (218, 140, 124, 16)
assert np.isnan(loaded).sum() == 0
assert (loaded < 0).sum()     == 0
print("\n✅ Submission ready: /kaggle/working/preds.npy")

Saved  → /kaggle/working/preds.npy
Shape  : (218, 140, 124, 16)
dtype  : float32
min/max: 0.0000 / 1014.4149
NaNs   : 0
Negs   : 0

✅ Submission ready: /kaggle/working/preds.npy
